In [6]:
import polars as pl
from sim_lib.sim_results import ManySimResults, SimResults
from pathlib import Path
import os
from tqdm import tqdm
import pandas as pd

In [2]:
# make sure that the new and old directories contain the same simulation params
assert set(os.listdir("data/july_2025")) == set(os.listdir("data/july_2025_COPY"))

In [3]:
combs = os.listdir("data/july_2025")

In [9]:
for c in tqdm(combs):
    p_new = Path("data/july_2025") / c
    p_old = Path("data/july_2025_COPY") / c

    new_sims = SimResults.load(p_new)
    old_sims = SimResults.load(p_old)

    # final_sims will hold the combined sim result summaries
    final_sims = SimResults(new_sims.sim_params)

    # we use beta0 to determine how many rows at the start are duplicated
    new_beta0 = new_sims.summary["beta1"]
    old_beta0 = old_sims.summary["beta1"]

    i = 0
    # Check if the rows are equal
    while new_beta0.iloc[i].equals(old_beta0.iloc[i]):
        i += 1

    # concat the summaries for each param
    for param in new_sims.summary.keys():
        # we need to clean new_summary because it could contain duplicate rows

        # there were some weird cases with the first 3 rows equal,
        # and it is from the first row of both being the same, so we just remove the first
        # row from both of them
        if i == 3:
            new_summary = new_sims.summary[param].iloc[i:]
            old_summary = old_sims.summary[param].iloc[1:]
        else:
            new_summary = new_sims.summary[param].iloc[i:]
            old_summary = old_sims.summary[param]

        new_summary = pd.concat([old_summary, new_summary])
        new_summary.reset_index(inplace=True)
        del new_summary["Unnamed: 0"]

        # if param == "beta0":
        #     assert (
        #         len(
        #             pl.concat(
        #                 [
        #                     pl.from_pandas(new_sims.summary[param]),
        #                     pl.from_pandas(old_sims.summary[param]),
        #                 ]
        #             ).unique()
        #         )
        #         == 100
        #     ), f"len of concat is not 100, {param, c}"

        assert len(new_summary) == 100, (
            f"len of new summary is {len(new_summary)}, {param, c}"
        )

        final_sims.summary[param] = new_summary

        # write the final sims to a new dir
        final_sims.save(f"data/july_2025_combined/{c}")

100%|██████████| 3024/3024 [31:07<00:00,  1.62it/s]  


In [30]:
old = "data/july_2025_COPY/sim_summary_902034851835405153"
new = "data/july_2025/sim_summary_902034851835405153"

o = SimResults.load(old).summary
n = SimResults.load(new).summary


In [31]:
o["p_aru01"]


,Unnamed: 0,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat,median
0,0,0.053,0.007,0.041,0.065,0.0,0.0,11888.0,18740.0,1.0,0.053
1,1,0.044,0.005,0.034,0.055,0.0,0.0,18919.0,24147.0,1.0,0.044
2,2,0.048,0.006,0.037,0.059,0.0,0.0,16636.0,23118.0,1.0,0.048
3,3,0.039,0.005,0.030,0.049,0.0,0.0,19943.0,24163.0,1.0,0.039
4,4,0.050,0.006,0.039,0.062,0.0,0.0,14778.0,22163.0,1.0,0.050
5,5,0.056,0.006,0.044,0.067,0.0,0.0,16023.0,20550.0,1.0,0.056
6,6,0.049,0.006,0.038,0.060,0.0,0.0,17348.0,23081.0,1.0,0.048
7,7,0.053,0.006,0.042,0.065,0.0,0.0,16553.0,23562.0,1.0,0.053
8,8,0.063,0.007,0.051,0.076,0.0,0.0,13968.0,19650.0,1.0,0.063
9,9,0.048,0.006,0.037,0.059,0.0,0.0,15552.0,21576.0,1.0,0.048


In [32]:
n["p_aru01"]

,Unnamed: 0,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat,median
0,0,0.053,0.007,0.041,0.065,0.0,0.0,11888.0,18740.0,1.0,0.053
1,1,0.044,0.005,0.034,0.055,0.0,0.0,18919.0,24147.0,1.0,0.044
2,2,0.048,0.006,0.037,0.059,0.0,0.0,16636.0,23118.0,1.0,0.048
3,3,0.051,0.006,0.041,0.062,0.0,0.0,17514.0,21309.0,1.0,0.051
4,4,0.044,0.006,0.034,0.055,0.0,0.0,16016.0,21504.0,1.0,0.044
5,5,0.047,0.006,0.035,0.058,0.0,0.0,12523.0,18251.0,1.0,0.046
6,6,0.048,0.005,0.038,0.058,0.0,0.0,18654.0,22873.0,1.0,0.048
7,7,0.053,0.006,0.042,0.064,0.0,0.0,18838.0,22914.0,1.0,0.053
8,8,0.059,0.006,0.048,0.071,0.0,0.0,13204.0,20104.0,1.0,0.059
9,9,0.051,0.006,0.041,0.062,0.0,0.0,18587.0,22371.0,1.0,0.051


In [42]:
pl.concat([pl.from_pandas(n["beta0"]), pl.from_pandas(o["beta0"])]).unique()

Unnamed: 0,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat,median
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
22,-0.839,0.278,-1.36,-0.317,0.002,0.001,20570.0,24469.0,1.0,-0.833
40,-0.681,0.261,-1.173,-0.19,0.002,0.001,26932.0,25894.0,1.0,-0.675
25,-1.114,0.261,-1.6,-0.621,0.002,0.001,28961.0,26997.0,1.0,-1.105
34,-1.126,0.286,-1.67,-0.597,0.002,0.001,23556.0,26036.0,1.0,-1.118
33,-1.268,0.311,-1.863,-0.697,0.002,0.001,19492.0,23381.0,1.0,-1.257
…,…,…,…,…,…,…,…,…,…,…
0,-0.555,0.259,-1.056,-0.082,0.002,0.001,25686.0,25434.0,1.0,-0.55
17,-1.26,0.308,-1.854,-0.699,0.002,0.002,21478.0,24089.0,1.0,-1.249
39,-0.935,0.288,-1.482,-0.403,0.002,0.001,21616.0,23546.0,1.0,-0.928
